# Urban Heat Island (UHI) Analysis Workflow — v2
## MSF Geo-Humanitarian Project — Tondo, Manila
**University of Salzburg · Z_GIS · Geo-Humanitarian Research Group**

---

### What's new in v2
- **Multi-sensor**: Landsat 8 + 9 combined (~8-day revisit) + MODIS Terra+Aqua LST (~2x daily, 1 km)
- **LST fusion**: Regression-based downscaling of MODIS → 30 m using Landsat spectral predictors
- **Monthly time-series**: Fused 30 m LST for every month, not just seasonal composites
- **Resolution-aware stack**: Each layer catalogued at native resolution — no blind resampling
- **All bug fixes**: GHSL band, WorldPop date, Healthsites path, `ee.Filter.Or`, pure folium

### How to use
> Edit **ONE cell** — `CONFIGURATION` — to change the study area. Everything else adapts automatically.

---


## 0. Google Earth Engine — Setup Guide

1. Register at [code.earthengine.google.com/register](https://code.earthengine.google.com/register)
2. Create a Cloud Project at [console.cloud.google.com](https://console.cloud.google.com/) → New Project
3. Enable the Earth Engine API at [APIs & Services → Library](https://console.cloud.google.com/apis/library/earthengine.googleapis.com)
4. Paste your Project ID in Cell 0B below

---


In [8]:
# ============================================================
# 0A. INSTALL DEPENDENCIES
# ============================================================
!pip install earthengine-api folium pandas matplotlib seaborn geopandas -q
print("All packages installed.")


All packages installed.


In [9]:
# ============================================================
# 0B. AUTHENTICATE & INITIALIZE GOOGLE EARTH ENGINE
# ============================================================
GEE_PROJECT = "uhi-msf-analysis"  # <-- Replace with your project ID

import ee
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

print(f"Earth Engine initialized with project: {GEE_PROJECT}")
print(f"Test: {ee.Date('2025-01-01').format('YYYY-MM-dd').getInfo()}")


Earth Engine initialized with project: uhi-msf-analysis
Test: 2025-01-01


In [10]:
# ============================================================
# 0C. IMPORTS
# ============================================================
import folium
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime

def add_ee_layer(m, ee_image, vis_params, name, shown=True):
    map_id = ee_image.getMapId(vis_params)
    folium.TileLayer(
        tiles=map_id['tile_fetcher'].url_format,
        attr='Google Earth Engine', name=name,
        overlay=True, control=True, show=shown
    ).add_to(m)

def make_map(center_lat, center_lon, zoom=14):
    return folium.Map(location=[center_lat, center_lon], zoom_start=zoom, tiles='OpenStreetMap')

print("All libraries loaded.")


All libraries loaded.


---
## 1. CONFIGURATION — *Edit this cell only*
---


In [11]:
# ============================================================
# 1. CONFIGURATION — *** CHANGE ONLY THIS CELL ***
# ============================================================

# ---------- AREA OF INTEREST ----------
AOI_BBOX = [120.950, 14.600, 120.985, 14.640]  # [west, south, east, north] Tondo, Manila
# AOI_ASSET = "FAO/GAUL/2015/level2"
# AOI_FILTER = ee.Filter.eq('ADM2_NAME', 'Manila')

# ---------- STUDY PERIOD ----------
YEAR_START = 2019
YEAR_END   = 2025

# ---------- SEASONS ----------
DRY_MONTHS = [12, 1, 2, 3, 4, 5]
WET_MONTHS = [6, 7, 8, 9, 10, 11]

# ---------- CLOUD COVER THRESHOLD (%) ----------
MAX_CLOUD_COVER = 30

# ---------- UHI REFERENCE BUFFER (meters) ----------
UHI_BUFFER_M = 5000

# ---------- VULNERABILITY INDEX WEIGHTS (sum = 1.0) ----------
W_LST    = 0.30
W_POP    = 0.25
W_BUILD  = 0.20
W_NDVI   = 0.15
W_HEALTH = 0.10

# ---------- OUTPUT ----------
PROJECT_NAME = "UHI_Tondo_Manila"

assert abs(W_LST + W_POP + W_BUILD + W_NDVI + W_HEALTH - 1.0) < 0.001, "Weights must sum to 1.0"
print(f"Config loaded: {PROJECT_NAME} | AOI: {AOI_BBOX} | {YEAR_START}-{YEAR_END}")


Config loaded: UHI_Tondo_Manila | AOI: [120.95, 14.6, 120.985, 14.64] | 2019-2025


## 2. Define Area of Interest

In [12]:
# ============================================================
# 2. BUILD AOI GEOMETRY
# ============================================================
try:
    aoi = ee.FeatureCollection(AOI_ASSET).filter(AOI_FILTER).geometry()
    print("AOI loaded from asset.")
except NameError:
    aoi = ee.Geometry.Rectangle(AOI_BBOX)
    print("AOI created from bounding box.")

aoi_buffer = aoi.buffer(UHI_BUFFER_M)
aoi_ring   = aoi_buffer.difference(aoi)

center_lat = (AOI_BBOX[1] + AOI_BBOX[3]) / 2
center_lon = (AOI_BBOX[0] + AOI_BBOX[2]) / 2

m = make_map(center_lat, center_lon, zoom=13)
folium.GeoJson(aoi.getInfo(), name='AOI',
               style_function=lambda x: {'color': 'red', 'weight': 2, 'fillOpacity': 0.1}).add_to(m)
folium.GeoJson(aoi_ring.getInfo(), name='Reference Ring',
               style_function=lambda x: {'color': 'blue', 'weight': 1, 'fillOpacity': 0.05}).add_to(m)
folium.LayerControl().add_to(m)
m


AOI created from bounding box.


## 3. Helper Functions (Landsat 8+9, MODIS, Fusion)

In [13]:
# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

# --- CLOUD MASKING (L8 & L9 Collection 2 share the same QA structure) ---
def mask_landsat_clouds(image):
    qa = image.select('QA_PIXEL')
    return image.updateMask(qa.bitwiseAnd(1 << 3).eq(0)).updateMask(qa.bitwiseAnd(1 << 4).eq(0))

# --- LST from Landsat (same scale/offset for L8 & L9 C02 L2) ---
def compute_lst_landsat(image):
    lst = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('LST')
    return image.addBands(lst)

# --- MODIS LST (Terra MOD11A1 & Aqua MYD11A1, 1 km, daily) ---
def compute_lst_modis(image):
    lst = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('LST_MOD')
    qa = image.select('QC_Day')
    good = qa.bitwiseAnd(3).eq(0)  # bits 0-1 = 00 means good quality
    return image.addBands(lst).updateMask(good)

# --- SPECTRAL INDICES ---
def compute_ndvi(image):
    return image.addBands(image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI'))

def compute_ndbi(image):
    return image.addBands(image.normalizedDifference(['SR_B6', 'SR_B5']).rename('NDBI'))

def compute_mndwi(image):
    return image.addBands(image.normalizedDifference(['SR_B3', 'SR_B6']).rename('MNDWI'))

# --- FILTERING ---
def filter_by_months(collection, months):
    def tag(image):
        m = ee.Number(image.date().get('month'))
        return image.set('in_season', ee.List(months).contains(m))
    return collection.map(tag).filter(ee.Filter.eq('in_season', True))

def add_time_props(image):
    d = image.date()
    return (image.set('year', d.get('year'))
                 .set('month', d.get('month'))
                 .set('doy', d.getRelative('day', 'year'))
                 .set('system_date', d.format('YYYY-MM-dd')))

# --- MIN-MAX NORMALIZE ---
def min_max_normalize(image, geometry, band_name=None, scale=30):
    if band_name:
        image = image.select(band_name)
    mm = image.reduceRegion(reducer=ee.Reducer.minMax(),
                            geometry=geometry, scale=scale, maxPixels=1e9)
    keys = mm.keys().getInfo()
    min_key = [k for k in keys if '_min' in k][0]
    max_key = [k for k in keys if '_max' in k][0]
    return image.subtract(ee.Number(mm.get(min_key))).divide(
        ee.Number(mm.get(max_key)).subtract(ee.Number(mm.get(min_key)))).clamp(0, 1)

print("Helper functions defined (L8, L9, MODIS, indices, normalization).")


Helper functions defined (L8, L9, MODIS, indices, normalization).


---
## 4. Data Acquisition — Landsat 8+9 & MODIS LST

In [14]:
# ============================================================
# 4. ACQUIRE LANDSAT 8+9 & MODIS LST
# ============================================================

# --- A. Landsat 9 ---
l9_col = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
          .filterBounds(aoi)
          .filterDate(f'{YEAR_START}-01-01', f'{YEAR_END}-12-31')
          .filter(ee.Filter.lt('CLOUD_COVER', MAX_CLOUD_COVER))
          .map(mask_landsat_clouds).map(compute_lst_landsat)
          .map(compute_ndvi).map(compute_ndbi).map(compute_mndwi)
          .map(add_time_props).map(lambda img: img.set('sensor', 'L9')))

# --- B. Landsat 8 ---
l8_col = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
          .filterBounds(aoi)
          .filterDate(f'{YEAR_START}-01-01', f'{YEAR_END}-12-31')
          .filter(ee.Filter.lt('CLOUD_COVER', MAX_CLOUD_COVER))
          .map(mask_landsat_clouds).map(compute_lst_landsat)
          .map(compute_ndvi).map(compute_ndbi).map(compute_mndwi)
          .map(add_time_props).map(lambda img: img.set('sensor', 'L8')))

# --- C. Merge Landsat 8 + 9 ---
landsat_col = l8_col.merge(l9_col).sort('system:time_start')

l8n = l8_col.size().getInfo()
l9n = l9_col.size().getInfo()
print(f"Landsat 8: {l8n} | Landsat 9: {l9n} | Combined: {l8n + l9n}")

dates = landsat_col.aggregate_array('system:time_start').getInfo()
if dates:
    dts = [datetime.utcfromtimestamp(d/1000).strftime('%Y-%m-%d') for d in dates]
    print(f"  Range: {min(dts)} -> {max(dts)}")

# --- D. MODIS LST (Terra + Aqua, 1 km, ~2x daily) ---
mod_terra = (ee.ImageCollection('MODIS/061/MOD11A1')
             .filterBounds(aoi)
             .filterDate(f'{YEAR_START}-01-01', f'{YEAR_END}-12-31')
             .map(compute_lst_modis))

mod_aqua = (ee.ImageCollection('MODIS/061/MYD11A1')
            .filterBounds(aoi)
            .filterDate(f'{YEAR_START}-01-01', f'{YEAR_END}-12-31')
            .map(compute_lst_modis))

modis_col = (mod_terra.merge(mod_aqua)
             .sort('system:time_start')
             .map(add_time_props)
             .map(lambda img: img.set('sensor', 'MODIS')))

modis_n = modis_col.size().getInfo()
print(f"MODIS LST (Terra+Aqua): {modis_n} scenes (1 km, ~2x daily)")


Landsat 8: 47 | Landsat 9: 27 | Combined: 74
  Range: 2019-01-04 -> 2025-12-30
MODIS LST (Terra+Aqua): 5084 scenes (1 km, ~2x daily)


## 4B. Resolution-Specific Spatio-Temporal Stack
Each data layer is catalogued and processed at its **native resolution** — no premature resampling.  
The fusion step is the only cross-resolution operation, done through regression.


In [15]:
# ============================================================
# 4B. RESOLUTION-SPECIFIC SPATIO-TEMPORAL STACK
# ============================================================
# +--------+------------+---------------------+----------------+
# | TIER   | RESOLUTION | LAYERS              | SOURCE         |
# +--------+------------+---------------------+----------------+
# | Tier 1 | 30 m       | LST, NDVI, NDBI,    | Landsat 8+9    |
# |        |            | MNDWI               |                |
# | Tier 2 | 10 m       | Land cover          | ESA WorldCover |
# |        |            | Built-up surface    | GHSL           |
# |        |            | Building footprints | Open Buildings |
# | Tier 3 | 100 m      | Population density  | WorldPop       |
# | Tier 4 | 1000 m     | LST (high-temporal) | MODIS T+A      |
# | Fused  | 30 m       | Downscaled LST      | MODIS+Landsat  |
# +--------+------------+---------------------+----------------+

stack_catalogue = pd.DataFrame([
    {'Tier': 1, 'Res_m': 30,   'Band': 'LST',       'Source': 'Landsat 8+9 C02 L2',   'Temporal': '~8-day (combined)'},
    {'Tier': 1, 'Res_m': 30,   'Band': 'NDVI',      'Source': 'Landsat 8+9 C02 L2',   'Temporal': '~8-day (combined)'},
    {'Tier': 1, 'Res_m': 30,   'Band': 'NDBI',      'Source': 'Landsat 8+9 C02 L2',   'Temporal': '~8-day (combined)'},
    {'Tier': 1, 'Res_m': 30,   'Band': 'MNDWI',     'Source': 'Landsat 8+9 C02 L2',   'Temporal': '~8-day (combined)'},
    {'Tier': 2, 'Res_m': 10,   'Band': 'LandCover', 'Source': 'ESA WorldCover v2',     'Temporal': 'Annual'},
    {'Tier': 2, 'Res_m': 10,   'Band': 'BuiltUp',   'Source': 'GHSL P2023A',           'Temporal': 'Epoch 2020'},
    {'Tier': 2, 'Res_m': 10,   'Band': 'Buildings',  'Source': 'Google Open Bldgs v3', 'Temporal': 'Snapshot'},
    {'Tier': 3, 'Res_m': 100,  'Band': 'Population', 'Source': 'WorldPop',             'Temporal': 'Annual'},
    {'Tier': 4, 'Res_m': 1000, 'Band': 'LST_MOD',    'Source': 'MODIS Terra+Aqua',    'Temporal': '~2x daily'},
    {'Tier': 'F','Res_m': 30,  'Band': 'LST_fused',  'Source': 'MODIS->30m regression','Temporal': 'Daily (downscaled)'},
])

print("Spatio-Temporal Stack Catalogue")
print("=" * 95)
print(stack_catalogue.to_string(index=False))

print("\nTemporal density (scenes per year):")
for yr in range(YEAR_START, YEAR_END + 1):
    ln = landsat_col.filter(ee.Filter.calendarRange(yr, yr, 'year')).size().getInfo()
    mn = modis_col.filter(ee.Filter.calendarRange(yr, yr, 'year')).size().getInfo()
    if ln > 0 or mn > 0:
        print(f"  {yr}: Landsat={ln}, MODIS={mn}")


Spatio-Temporal Stack Catalogue
Tier  Res_m       Band                Source           Temporal
   1     30        LST    Landsat 8+9 C02 L2  ~8-day (combined)
   1     30       NDVI    Landsat 8+9 C02 L2  ~8-day (combined)
   1     30       NDBI    Landsat 8+9 C02 L2  ~8-day (combined)
   1     30      MNDWI    Landsat 8+9 C02 L2  ~8-day (combined)
   2     10  LandCover     ESA WorldCover v2             Annual
   2     10    BuiltUp           GHSL P2023A         Epoch 2020
   2     10  Buildings  Google Open Bldgs v3           Snapshot
   3    100 Population              WorldPop             Annual
   4   1000    LST_MOD      MODIS Terra+Aqua          ~2x daily
   F     30  LST_fused MODIS->30m regression Daily (downscaled)

Temporal density (scenes per year):
  2019: Landsat=11, MODIS=730
  2020: Landsat=8, MODIS=732
  2021: Landsat=5, MODIS=730
  2022: Landsat=14, MODIS=702
  2023: Landsat=13, MODIS=730
  2024: Landsat=15, MODIS=732
  2025: Landsat=8, MODIS=728


## 5. Seasonal & Annual Composites (Landsat + MODIS)

In [16]:
# ============================================================
# 5. SEASONAL & ANNUAL COMPOSITES
# ============================================================

# Landsat composites (30 m)
dry_landsat = filter_by_months(landsat_col, DRY_MONTHS).select(['LST','NDVI','NDBI','MNDWI']).median().clip(aoi_buffer)
wet_landsat = filter_by_months(landsat_col, WET_MONTHS).select(['LST','NDVI','NDBI','MNDWI']).median().clip(aoi_buffer)
ann_landsat = landsat_col.select(['LST','NDVI','NDBI','MNDWI']).median().clip(aoi_buffer)

# MODIS composites (1 km)
dry_mod = filter_by_months(modis_col, DRY_MONTHS).select('LST_MOD').median().clip(aoi_buffer)
wet_mod = filter_by_months(modis_col, WET_MONTHS).select('LST_MOD').median().clip(aoi_buffer)
ann_mod = modis_col.select('LST_MOD').median().clip(aoi_buffer)

dry_l = filter_by_months(landsat_col, DRY_MONTHS).size().getInfo()
wet_l = filter_by_months(landsat_col, WET_MONTHS).size().getInfo()
dry_m = filter_by_months(modis_col, DRY_MONTHS).size().getInfo()
wet_m = filter_by_months(modis_col, WET_MONTHS).size().getInfo()
print(f"Composites built:")
print(f"  Landsat — Dry: {dry_l}, Wet: {wet_l}")
print(f"  MODIS   — Dry: {dry_m}, Wet: {wet_m}")


Composites built:
  Landsat — Dry: 55, Wet: 19
  MODIS   — Dry: 2534, Wet: 2550


## 5B. LST Fusion: MODIS (1 km) → 30 m
Regression-based spatial downscaling:
1. Aggregate Landsat NDVI / NDBI / MNDWI to 1 km to match MODIS
2. Fit multivariate linear regression: `MODIS_LST = a + b*NDVI + c*NDBI + d*MNDWI`
3. Apply coefficients to 30 m Landsat predictors → fused 30 m LST
4. Add residual correction (bilinear-interpolated 1 km residual) to preserve MODIS absolute accuracy


In [17]:
# ============================================================
# 5B. FUSION FUNCTION
# ============================================================

def fuse_lst(landsat_comp, mod_comp, geometry, label=""):
    """Downscale MODIS LST to 30 m using Landsat spectral indices."""

    predictors_30m = landsat_comp.select(['NDVI', 'NDBI', 'MNDWI']).clip(geometry)

    # Aggregate Landsat predictors to 1 km to match MODIS
    predictors_1km = (predictors_30m
        .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=2048)
        .reproject(crs='EPSG:4326', scale=1000))

    mod_lst = mod_comp.select('LST_MOD').clip(geometry)
    training_stack = mod_lst.addBands(predictors_1km)

    training = training_stack.sample(
        region=geometry, scale=1000, numPixels=500, seed=42, geometries=False)

    n = training.size().getInfo()
    if n < 10:
        print(f"  {label}: {n} MODIS samples — too few, using Landsat LST.")
        return landsat_comp.select('LST').rename('LST_fused')

    # Add constant for intercept
    def add_constant(feat):
        return feat.set('constant', 1)
    training = training.map(add_constant)

    # linearRegression: numX includes the constant
    regression = training.reduceColumns(
        reducer=ee.Reducer.linearRegression(numX=4, numY=1),
        selectors=['constant', 'NDVI', 'NDBI', 'MNDWI', 'LST_MOD'])

    coeffs = ee.Array(regression.get('coefficients')).project([0]).toList()
    b0 = ee.Number(coeffs.get(0))  # intercept
    b1 = ee.Number(coeffs.get(1))  # NDVI
    b2 = ee.Number(coeffs.get(2))  # NDBI
    b3 = ee.Number(coeffs.get(3))  # MNDWI

    # Apply regression to 30 m predictors
    fused = (predictors_30m.select('NDVI').multiply(b1)
             .add(predictors_30m.select('NDBI').multiply(b2))
             .add(predictors_30m.select('MNDWI').multiply(b3))
             .add(b0).rename('LST_fused'))

    # Residual correction: observed MODIS - predicted at 1 km
    pred_1km = (predictors_1km.select('NDVI').multiply(b1)
                .add(predictors_1km.select('NDBI').multiply(b2))
                .add(predictors_1km.select('MNDWI').multiply(b3))
                .add(b0))
    residual = mod_lst.subtract(pred_1km)
    residual_30m = residual.resample('bilinear').reproject(crs='EPSG:4326', scale=30)
    fused = fused.add(residual_30m).rename('LST_fused')

    print(f"  {label}: n={n}, b0={b0.getInfo():.2f}, "
          f"b_NDVI={b1.getInfo():.3f}, b_NDBI={b2.getInfo():.3f}, b_MNDWI={b3.getInfo():.3f}")
    return fused

print("Fusion function defined.")


Fusion function defined.


In [18]:
# ============================================================
# 5C. EXECUTE SEASONAL & ANNUAL FUSION
# ============================================================

print("Running regression-based LST fusion (MODIS 1 km -> 30 m)...")
fused_dry = fuse_lst(dry_landsat, dry_mod, aoi_buffer, "Dry season")
fused_wet = fuse_lst(wet_landsat, wet_mod, aoi_buffer, "Wet season")
fused_ann = fuse_lst(ann_landsat, ann_mod, aoi_buffer, "Annual")

# Final composites: Landsat bands + fused LST
dry_composite    = dry_landsat.addBands(fused_dry)
wet_composite    = wet_landsat.addBands(fused_wet)
annual_composite = ann_landsat.addBands(fused_ann)

print("\nFused composites ready:")
print("  LST       -> Landsat-native 30 m")
print("  LST_fused -> MODIS-downscaled 30 m (regression + residual correction)")
print("  NDVI, NDBI, MNDWI -> Landsat 30 m")


Running regression-based LST fusion (MODIS 1 km -> 30 m)...


EEException: Image.reduceResolution: The input to reduceResolution does not have a valid default projection. Use setDefaultProjection() first.

## 5D. Monthly LST Time-Series Fusion (30 m)
Produces a fused 30 m LST image for **every month** in the study period.  
Where a Landsat composite exists, it's used as the predictor base.  
MODIS's ~2x daily coverage fills the temporal gaps Landsat can't cover.


In [ ]:
# ============================================================
# 5D. MONTHLY LST TIME-SERIES (FUSED 30 m)
# ============================================================

monthly_results = []

print("Building monthly fused LST time-series...")
print(f"{'Month':<10} {'L8+L9':>6} {'MODIS':>6} {'Method':<22} {'Mean LST':>10}")
print("-" * 62)

for yr in range(YEAR_START, YEAR_END + 1):
    for mo in range(1, 13):
        start = f'{yr}-{mo:02d}-01'
        end_date = ee.Date(start).advance(1, 'month')

        l_mo = landsat_col.filterDate(start, end_date).select(['LST','NDVI','NDBI','MNDWI'])
        m_mo = modis_col.filterDate(start, end_date).select('LST_MOD')

        ln = l_mo.size().getInfo()
        mn = m_mo.size().getInfo()

        if ln == 0 and mn == 0:
            continue

        if ln >= 1 and mn >= 5:
            l_comp = l_mo.median().clip(aoi_buffer)
            m_comp = m_mo.median().clip(aoi_buffer)
            fused = fuse_lst(l_comp, m_comp, aoi_buffer, f"{yr}-{mo:02d}")
            method = "Fused (MODIS->30m)"
        elif ln >= 1:
            fused = l_mo.select('LST').median().clip(aoi_buffer).rename('LST_fused')
            method = "Landsat only"
        else:
            fused = (m_mo.select('LST_MOD').median().clip(aoi_buffer)
                     .resample('bilinear').reproject(crs='EPSG:4326', scale=30)
                     .rename('LST_fused'))
            method = "MODIS resampled"

        mean_lst = fused.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=aoi,
            scale=30, maxPixels=1e9).get('LST_fused').getInfo()

        monthly_results.append({
            'Year': yr, 'Month': mo, 'Date': f'{yr}-{mo:02d}',
            'Landsat_scenes': ln, 'MODIS_scenes': mn,
            'Method': method,
            'Mean_LST_fused': round(mean_lst, 2) if mean_lst else None
        })

        if mean_lst:
            print(f"  {yr}-{mo:02d}   {ln:6d} {mn:6d} {method:<22} {mean_lst:10.2f} C")

monthly_df = pd.DataFrame(monthly_results)
print(f"\nMonthly time-series: {len(monthly_df)} months with data.")


In [ ]:
# ============================================================
# 5E. MONTHLY LST TIME-SERIES PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(14, 5))

colors = {'Fused (MODIS->30m)': '#d73027', 'Landsat only': '#4575b4', 'MODIS resampled': '#fdae61'}
for method, grp in monthly_df.groupby('Method'):
    ax.plot(grp['Date'], grp['Mean_LST_fused'], 'o-', color=colors.get(method, 'gray'),
            label=method, markersize=4, linewidth=1)

ax.set_xlabel('Month')
ax.set_ylabel('Mean Fused LST (C)')
ax.set_title(f'{PROJECT_NAME} — Monthly LST Time-Series (30 m fused)')
ax.legend()
ax.set_xticks(ax.get_xticks()[::3])
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('monthly_lst_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: monthly_lst_timeseries.png")


## 5F. Fusion Validation — Landsat vs Fused vs MODIS

In [ ]:
# ============================================================
# 5F. FUSION VALIDATION
# ============================================================

val_stack = (annual_composite.select('LST').rename('LST_Landsat')
             .addBands(annual_composite.select('LST_fused').rename('LST_Fused')))

val_samples = val_stack.sample(region=aoi, scale=30, numPixels=1000,
                                seed=42, geometries=False).getInfo()
val_df = pd.DataFrame([f['properties'] for f in val_samples['features']]).dropna()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'{PROJECT_NAME} — LST Fusion Validation', fontsize=14, fontweight='bold')

axes[0].scatter(val_df['LST_Landsat'], val_df['LST_Fused'], alpha=0.3, s=8, c='#d73027')
lims = [val_df[['LST_Landsat','LST_Fused']].min().min()-1,
        val_df[['LST_Landsat','LST_Fused']].max().max()+1]
axes[0].plot(lims, lims, 'k--', lw=1)
axes[0].set_xlabel('Landsat LST (C)'); axes[0].set_ylabel('Fused LST (C)')
axes[0].set_title('Landsat vs Fused')

axes[1].hist(val_df['LST_Landsat'], bins=40, alpha=0.6, color='#4575b4', label='Landsat', edgecolor='white')
axes[1].hist(val_df['LST_Fused'], bins=40, alpha=0.6, color='#d73027', label='Fused', edgecolor='white')
axes[1].legend(); axes[1].set_xlabel('LST (C)'); axes[1].set_title('Distribution Comparison')

diff = val_df['LST_Fused'] - val_df['LST_Landsat']
axes[2].hist(diff, bins=40, color='#fdae61', alpha=0.7, edgecolor='white')
axes[2].axvline(0, color='black', ls='--')
axes[2].set_xlabel('Fused - Landsat (C)')
axes[2].set_title(f'Residual (mean={diff.mean():.2f}, std={diff.std():.2f})')

plt.tight_layout()
plt.savefig('lst_fusion_validation.png', dpi=150, bbox_inches='tight')
plt.show()

# Map comparison
lst_pal = ['313695','4575b4','74add1','abd9e9','e0f3f8','ffffbf','fee090','fdae61','f46d43','d73027','a50026']
m_fus = make_map(center_lat, center_lon, zoom=14)
add_ee_layer(m_fus, annual_composite.select('LST').clip(aoi), {'min':25,'max':45,'palette':lst_pal}, 'Landsat LST (30m)')
add_ee_layer(m_fus, annual_composite.select('LST_fused').clip(aoi), {'min':25,'max':45,'palette':lst_pal}, 'Fused LST (30m)', shown=False)
add_ee_layer(m_fus, ann_mod.clip(aoi), {'min':25,'max':45,'palette':lst_pal}, 'MODIS LST (1km)', shown=False)
folium.LayerControl().add_to(m_fus)
m_fus


## 6. UHI Intensity (Fused LST + Landsat comparison)

In [ ]:
# ============================================================
# 6. UHI INTENSITY
# ============================================================

def compute_uhi(composite, lst_band, label):
    lst = composite.select(lst_band)
    ref = ee.Number(lst.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi_ring,
        scale=30, maxPixels=1e9).get(lst_band))
    uhi = lst.subtract(ref).rename('UHI_intensity')
    print(f"  {label} ref LST: {ref.getInfo():.2f} C")
    return uhi

print("UHI from fused LST:")
uhi_dry    = compute_uhi(dry_composite, 'LST_fused', 'Dry')
uhi_wet    = compute_uhi(wet_composite, 'LST_fused', 'Wet')
uhi_annual = compute_uhi(annual_composite, 'LST_fused', 'Annual')

print("\nUHI from Landsat-only LST (comparison):")
uhi_dry_l    = compute_uhi(dry_composite, 'LST', 'Dry')
uhi_wet_l    = compute_uhi(wet_composite, 'LST', 'Wet')
uhi_annual_l = compute_uhi(annual_composite, 'LST', 'Annual')


## 7. Interactive Maps — LST & UHI Hotspots

In [ ]:
# ============================================================
# 7. INTERACTIVE MAP — LST & UHI
# ============================================================
lst_vis = {'min': 25, 'max': 45, 'palette': [
    '313695','4575b4','74add1','abd9e9','e0f3f8',
    'ffffbf','fee090','fdae61','f46d43','d73027','a50026']}
uhi_vis = {'min': -3, 'max': 6, 'palette': [
    '2166ac','67a9cf','d1e5f0','fddbc7','ef8a62','b2182b']}
ndvi_vis = {'min': -0.1, 'max': 0.6, 'palette': [
    'FFFFFF','CE7E45','DF923D','F1B555','FCD163',
    '99B718','74A901','66A000','529400','3E8601','207401','056201']}

m2 = make_map(center_lat, center_lon, zoom=14)
add_ee_layer(m2, dry_composite.select('LST_fused').clip(aoi), lst_vis, 'Fused LST — Dry')
add_ee_layer(m2, wet_composite.select('LST_fused').clip(aoi), lst_vis, 'Fused LST — Wet', shown=False)
add_ee_layer(m2, annual_composite.select('LST_fused').clip(aoi), lst_vis, 'Fused LST — Annual', shown=False)
add_ee_layer(m2, annual_composite.select('LST').clip(aoi), lst_vis, 'Landsat LST — Annual', shown=False)
add_ee_layer(m2, uhi_dry.clip(aoi), uhi_vis, 'UHI — Dry (fused)')
add_ee_layer(m2, uhi_wet.clip(aoi), uhi_vis, 'UHI — Wet (fused)', shown=False)
add_ee_layer(m2, uhi_annual.clip(aoi), uhi_vis, 'UHI — Annual (fused)', shown=False)
add_ee_layer(m2, annual_composite.select('NDVI').clip(aoi), ndvi_vis, 'NDVI — Annual', shown=False)
folium.LayerControl().add_to(m2)
m2


---
## 8. Ancillary Data Layers

In [ ]:
# ============================================================
# 8. ANCILLARY DATA
# ============================================================

# A. Population density — WorldPop (latest available year)
pop = (ee.ImageCollection("WorldPop/GP/100m/pop")
       .filterBounds(aoi)
       .sort('system:time_start', False)
       .first()
       .select('population')
       .clip(aoi))
pop_year = ee.Date(pop.get('system:time_start')).format('YYYY').getInfo()
print(f"Population loaded (WorldPop, year: {pop_year}).")

# B. Built-up surface — GHSL 2020 (correct band: built_surface)
ghsl = (ee.Image("JRC/GHSL/P2023A/GHS_BUILT_S/2020")
        .select('built_surface').clip(aoi))
print("GHSL built-up surface loaded.")

# C. ESA WorldCover 2021
esa_lc = ee.ImageCollection("ESA/WorldCover/v200").first().clip(aoi)
print("ESA WorldCover 2021 loaded.")

# D. Health facility proximity — Healthsites (sat-io)
health_dist = None
try:
    health_nodes = ee.FeatureCollection("projects/sat-io/open-datasets/health-site-node").filterBounds(aoi)
    health_ways  = ee.FeatureCollection("projects/sat-io/open-datasets/health-site-way").filterBounds(aoi)
    health_all   = health_nodes.merge(health_ways)
    hc_count = health_all.size().getInfo()
    if hc_count > 0:
        health_dist = health_all.distance(5000).clip(aoi).rename('health_dist')
        print(f"Health facilities loaded from Healthsites ({hc_count} features).")
    else:
        raise Exception("No health facilities in AOI")
except Exception as e:
    print(f"Healthsites not available: {e}")
    health_dist = ee.Image.constant(0).rename('health_dist').clip(aoi).float()
    print("Using placeholder — set W_HEALTH=0 or upload local health data.")

# E. Google Open Buildings
try:
    open_buildings = (ee.FeatureCollection("GOOGLE/Research/open-buildings/v3/polygons")
                      .filterBounds(aoi))
    bldg_count = open_buildings.size().getInfo()
    bldg_raster = open_buildings.reduceToImage(
        properties=['confidence'], reducer=ee.Reducer.count()
    ).rename('building_count').clip(aoi)
    print(f"Google Open Buildings loaded ({bldg_count} footprints).")
except Exception:
    print("Open Buildings not available — using GHSL as fallback.")
    bldg_raster = ghsl.rename('building_count')


---
## 9. Statistical Analysis & Exploratory Data Investigation

In [ ]:
# ============================================================
# 9A. ZONAL STATISTICS (fused + Landsat)
# ============================================================

def get_zonal_stats(image, band, geometry, scale=30):
    return image.select(band).reduceRegion(
        reducer=ee.Reducer.mean()
                  .combine(ee.Reducer.minMax(), sharedInputs=True)
                  .combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=geometry, scale=scale, maxPixels=1e9).getInfo()

rows = []
for season, comp in [('Annual', annual_composite), ('Dry', dry_composite), ('Wet', wet_composite)]:
    for band, label in [('LST_fused', 'Fused'), ('LST', 'Landsat')]:
        s = get_zonal_stats(comp, band, aoi)
        rows.append({
            'Season': season, 'Source': label,
            'Mean (C)': round(s.get(f'{band}_mean', 0), 2),
            'Min (C)':  round(s.get(f'{band}_min', 0), 2),
            'Max (C)':  round(s.get(f'{band}_max', 0), 2),
            'Std (C)':  round(s.get(f'{band}_stdDev', 0), 2)})

stats_df = pd.DataFrame(rows)
print("LST Summary Statistics (Fused vs Landsat)")
print("=" * 70)
print(stats_df.to_string(index=False))


In [ ]:
# ============================================================
# 9B. PIXEL SAMPLING
# ============================================================

stack = (annual_composite.select('LST_fused').rename('LST')
         .addBands(annual_composite.select('NDVI'))
         .addBands(annual_composite.select('NDBI'))
         .addBands(annual_composite.select('MNDWI'))
         .addBands(pop.rename('POP'))
         .addBands(ghsl.rename('BUILT')))

samples = stack.sample(region=aoi, scale=30, numPixels=2000,
                       seed=42, geometries=False).getInfo()
rows = [f['properties'] for f in samples['features']]
df = pd.DataFrame(rows).dropna()
print(f"Sampled {len(df)} pixels.")
print(df.describe().round(2))


In [ ]:
# ============================================================
# 9C. CORRELATION ANALYSIS
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(f'{PROJECT_NAME} — Fused LST Correlation Analysis', fontsize=14, fontweight='bold')

corr = df[['LST','NDVI','NDBI','MNDWI','POP','BUILT']].corr()
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, ax=axes[0,0], fmt='.2f')
axes[0,0].set_title('Correlation Matrix')

for ax, col, c, t in [(axes[0,1],'NDVI','green','LST vs NDVI'),
                       (axes[0,2],'NDBI','orange','LST vs NDBI'),
                       (axes[1,0],'POP','purple','LST vs Population'),
                       (axes[1,1],'BUILT','brown','LST vs Built-up')]:
    ax.scatter(df[col], df['LST'], alpha=0.3, s=5, c=c)
    ax.set_xlabel(col); ax.set_ylabel('LST (C)'); ax.set_title(t)

axes[1,2].hist(df['LST'], bins=40, color='#d73027', alpha=0.7, edgecolor='white')
axes[1,2].set_xlabel('LST (C)'); axes[1,2].set_ylabel('Count'); axes[1,2].set_title('LST Distribution')

plt.tight_layout()
plt.savefig('lst_correlation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Temporal Trends — Monthly + Yearly

In [ ]:
# ============================================================
# 10. TEMPORAL TRENDS
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={'height_ratios': [2, 1]})

# Monthly
ax1 = axes[0]
valid = monthly_df.dropna(subset=['Mean_LST_fused'])
ax1.plot(valid['Date'], valid['Mean_LST_fused'], 'o-', color='#d73027', markersize=3, lw=1)
ax1.set_ylabel('Mean Fused LST (C)')
ax1.set_title(f'{PROJECT_NAME} — Monthly Mean LST (30 m fused)')
ax1.set_xticks(ax1.get_xticks()[::6])
ax1.tick_params(axis='x', rotation=45)

# Yearly
yearly = valid.groupby('Year').agg(
    Mean_LST=('Mean_LST_fused', 'mean'),
    Months=('Mean_LST_fused', 'count')).reset_index()

ax2 = axes[1]
ax2.bar(yearly['Year'].astype(str), yearly['Mean_LST'], color='#fc8d59', edgecolor='white')
ax2.set_ylabel('Mean LST (C)')
ax2.set_title('Yearly Mean LST')
for _, r in yearly.iterrows():
    ax2.annotate(f"{r['Mean_LST']:.1f}", (str(int(r['Year'])), r['Mean_LST']),
                 ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('lst_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print(yearly.to_string(index=False))


---
## 11. Heat-Exposure Vulnerability Index (Fused LST)
Each layer is min-max normalized to [0, 1], then weighted and summed.

In [ ]:
# ============================================================
# 11. COMPOSITE VULNERABILITY INDEX (using fused LST)
# ============================================================

lst_norm   = min_max_normalize(annual_composite, aoi, 'LST_fused').rename('LST_norm')
pop_norm   = min_max_normalize(pop, aoi).rename('POP_norm')
built_norm = min_max_normalize(ghsl, aoi).rename('BUILT_norm')
ndvi_norm  = min_max_normalize(annual_composite, aoi, 'NDVI')
ndvi_inv   = ee.Image.constant(1).subtract(ndvi_norm).rename('NDVI_inv_norm')
health_norm = min_max_normalize(health_dist, aoi).rename('HEALTH_norm')

vulnerability = (lst_norm.multiply(W_LST)
                 .add(pop_norm.multiply(W_POP))
                 .add(built_norm.multiply(W_BUILD))
                 .add(ndvi_inv.multiply(W_NDVI))
                 .add(health_norm.multiply(W_HEALTH))
                ).rename('vulnerability').clip(aoi)

print(f"Vulnerability index computed (fused LST).")
print(f"  Weights: LST={W_LST}, POP={W_POP}, BUILD={W_BUILD}, NDVI={W_NDVI}, HEALTH={W_HEALTH}")


In [ ]:
# ============================================================
# 12. MAP — VULNERABILITY INDEX
# ============================================================
vuln_vis = {'min': 0, 'max': 1, 'palette': [
    '1a9850','91cf60','d9ef8b','ffffbf','fee08b','fc8d59','d73027']}

m3 = make_map(center_lat, center_lon, zoom=14)
add_ee_layer(m3, vulnerability, vuln_vis, 'Heat Vulnerability Index')
add_ee_layer(m3, lst_norm.clip(aoi),   {'min':0,'max':1,'palette':['blue','yellow','red']}, 'LST (norm)', shown=False)
add_ee_layer(m3, pop_norm.clip(aoi),   {'min':0,'max':1,'palette':['white','purple']}, 'Population (norm)', shown=False)
add_ee_layer(m3, ndvi_inv.clip(aoi),   {'min':0,'max':1,'palette':['green','white','brown']}, 'Vegetation Deficit', shown=False)
add_ee_layer(m3, built_norm.clip(aoi), {'min':0,'max':1,'palette':['white','grey','black']}, 'Built-up (norm)', shown=False)
folium.LayerControl().add_to(m3)
m3


## 12. Barangay-Level Vulnerability Ranking

In [ ]:
# ============================================================
# 13. BARANGAY-LEVEL RANKING
# ============================================================
try:
    barangays = (ee.FeatureCollection("projects/sat-io/open-datasets/geoboundaries/CGAZ_ADM4")
                 .filterBounds(aoi))
    brgy_count = barangays.size().getInfo()
    if brgy_count == 0:
        raise Exception("No features found")

    brgy_vuln = vulnerability.reduceRegions(
        collection=barangays,
        reducer=ee.Reducer.mean().setOutputs(['vuln_mean']), scale=30)
    brgy_lst = annual_composite.select('LST_fused').reduceRegions(
        collection=barangays,
        reducer=ee.Reducer.mean().setOutputs(['lst_mean']), scale=30)

    vuln_list = brgy_vuln.getInfo()['features']
    lst_list  = brgy_lst.getInfo()['features']
    ranking = []
    for v, l in zip(vuln_list, lst_list):
        name = v['properties'].get('shapeName', v['properties'].get('NAME_4', 'Unknown'))
        ranking.append({
            'Barangay': name,
            'Vulnerability': round(v['properties'].get('vuln_mean', 0), 3),
            'Mean LST (C)': round(l['properties'].get('lst_mean', 0), 2)})
    rank_df = pd.DataFrame(ranking).sort_values('Vulnerability', ascending=False)
    print("Barangay Vulnerability Ranking")
    print("=" * 50)
    print(rank_df.to_string(index=False))
except Exception as e:
    print(f"Barangay boundaries not available: {e}")
    print("  Upload a boundary shapefile as a GEE asset and update the path above.")


---
## 13. Export Results to Google Drive

In [ ]:
# ============================================================
# 14. EXPORT TO GOOGLE DRIVE
# ============================================================
export_params = dict(region=aoi, scale=30, maxPixels=1e9, crs='EPSG:4326')

exports = [
    (vulnerability.toFloat(),                           'vulnerability_index'),
    (annual_composite.select('LST_fused').toFloat(),    'annual_LST_fused'),
    (annual_composite.select('LST').toFloat(),          'annual_LST_landsat'),
    (uhi_dry.toFloat(),                                 'UHI_dry_fused'),
    (uhi_annual.toFloat(),                              'UHI_annual_fused'),
]

for img, name in exports:
    task = ee.batch.Export.image.toDrive(
        image=img, description=f'{PROJECT_NAME}_{name}',
        folder=PROJECT_NAME, **export_params)
    task.start()
    print(f"Export: {PROJECT_NAME}_{name}")

print(f"\nCheck progress: https://code.earthengine.google.com/tasks")

stats_df.to_csv('lst_summary_statistics.csv', index=False)
monthly_df.to_csv('monthly_lst_timeseries.csv', index=False)
if 'rank_df' in dir():
    rank_df.to_csv('barangay_vulnerability_ranking.csv', index=False)
print("CSV files saved.")


---
## 14. [Phase 3] Cooling Site Identification *(lower priority)*

In [ ]:
# ============================================================
# 15. COOLING SITE IDENTIFICATION
# ============================================================
cool_classes = esa_lc.eq(10).Or(esa_lc.eq(20)).Or(esa_lc.eq(30))
cool_mask = cool_classes.selfMask().rename('cool_areas')

vuln_p75 = vulnerability.reduceRegion(
    reducer=ee.Reducer.percentile([75]),
    geometry=aoi, scale=30, maxPixels=1e9).get('vulnerability')
high_vuln = vulnerability.gte(ee.Number(vuln_p75)).selfMask()

m4 = make_map(center_lat, center_lon, zoom=14)
add_ee_layer(m4, vulnerability, vuln_vis, 'Vulnerability Index')
add_ee_layer(m4, high_vuln.clip(aoi), {'min':0,'max':1,'palette':['red']}, 'High Vulnerability Zones')
add_ee_layer(m4, cool_mask.clip(aoi), {'min':0,'max':1,'palette':['00ff00']}, 'Potential Cool Spaces', shown=False)
folium.LayerControl().add_to(m4)
m4


---
## Summary & Next Steps

### Outputs (v2):
1. **Multi-sensor composites** — Landsat 8+9 (30 m, ~8-day) + MODIS Terra+Aqua (1 km, ~2x daily)
2. **Fused LST** — regression-based downscaling of MODIS -> 30 m with residual correction
3. **Monthly LST time-series** — 30 m fused LST for every month in the study period
4. **Fusion validation** — scatter, distribution, residual analysis + map comparison
5. **Seasonal UHI intensity maps** — from fused LST with Landsat comparison
6. **Statistical analysis** — zonal stats (fused vs Landsat), correlations, temporal trends
7. **Heat-exposure vulnerability index** — composite map using fused LST
8. **Barangay-level ranking** + cooling site candidates
9. **Exported GeoTIFFs** — fused LST, Landsat LST, UHI, vulnerability to Google Drive
10. **Resolution-aware stack catalogue** — all layers documented at native resolution

### To adapt for a new study area:
1. Change `AOI_BBOX` in Configuration
2. Adjust dates / seasons if needed
3. Re-run all cells

### Methodology note
MODIS Terra+Aqua was chosen over Sentinel-3 SLSTR because MODIS has a verified GEE collection
with consistent QA flags, a longer archive (covering the full 2019-2025 study period), and
~2x daily revisit (Terra + Aqua combined). The 1 km resolution is bridged to 30 m through
regression-based downscaling using Landsat spectral indices as spatial predictors.

---
*MSF Geo-Humanitarian Project · University of Salzburg Z_GIS*
